# 08 · Master Pipeline Kaggle

Pipeline completo: prepara scripts a partir do checkout do GitHub, detecta GPU, configura Google Drive para persistência/backup, instala ComfyUI com output local no SSD, sincroniza modelos selecionados manualmente, inicia ComfyUI no SSD local com health check e oferece push inicial condicional.


In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "colab-pipeline"
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"
OUTPUT_DIR = COMFYUI_DIR / "output"
MODELS_DIR = COMFYUI_DIR / "models"
DATASET = "automamermaid/comfydocs"
DRIVE_BASE = "Automa/ComfyUI"

print("=== GPU / DISCO ===")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)


In [ ]:
# Sincroniza sempre o repositório como fonte da verdade e copia scripts para execução
import shutil

if REPO_DIR.exists():
    print("[INFO] Atualizando repositório via git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print("[INFO] Clonando repositório...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Copiar scripts atualizados do checkout garantindo que versões órfãs sejam eliminadas
if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(REPO_DIR / "scripts", SCRIPTS_DIR)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("[INFO] Scripts atualizados disponíveis:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))


In [ ]:
# Detecção e validação de GPU NVIDIA
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative Accelerator → GPU no Kaggle antes de continuar.")


In [ ]:
# Configuração do Google Drive (rclone + service account) SOMENTE para persistência/backup
from kaggle_drive_sync import get_drive_path, setup_rclone_kaggle, test_drive_connection

print("=" * 60)
print("CONFIGURANDO GOOGLE DRIVE (PERSISTÊNCIA/BACKUP)")
print("=" * 60)

DRIVE_AVAILABLE = False
try:
    test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
    if test_res["status"] == "pass":
        DRIVE_AVAILABLE = True
        DRIVE_PATH = Path(test_res["drive_path"])
        print(f"[INFO] Google Drive pronto para sync em: {DRIVE_PATH}")
    else:
        print(f"[WARN] Google Drive não pôde ser montado: {test_res.get('error')}")
except Exception as e:
    print(f"[WARN] Falha ao configurar Google Drive: {e}")
    print("[INFO] O ComfyUI funcionará normalmente gerando no SSD local.")


In [ ]:
# Instala/atualiza ComfyUI + custom nodes usando output SEMPRE local no SSD
from comfyui_setup import setup_comfyui

CUSTOM_NODES = [
    "ltdrdata/ComfyUI-Manager",
    "cubiq/ComfyUI_essentials",
]

OUTPUT_DIR = COMFYUI_DIR / "output"

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=OUTPUT_DIR,
)
print(f"[INFO] ComfyUI configurado com output local no SSD: {OUTPUT_DIR}")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from kaggle_sync import get_dataset_files, get_dataset_files_details, filter_dataset_files, sync_dataset_to_local, MODEL_CATEGORIES

def choose_dataset_files(dataset=DATASET, preselected_categories=None):
    """Abre caixa de seleção SelectMultiple mostrando nome, categoria/path e tamanho."""
    try:
        details = get_dataset_files_details(dataset)
    except Exception as e:
        print(f"[WARN] Não foi possível obter detalhes dos arquivos: {e}")
        details = []

    if details:
        if preselected_categories:
            candidates = [d for d in details if d["category"] in preselected_categories]
        else:
            candidates = details
        if not candidates:
            candidates = details
        options = [(f"{d['name']}  [{d['category']}/]  ({d['size']})", d["path"]) for d in candidates]
        paths = [d["path"] for d in candidates]
    else:
        raw_files = get_dataset_files(dataset)
        if not raw_files:
            raise RuntimeError("O Dataset não possui arquivos ou não pôde ser listado.")
        filtered = filter_dataset_files(raw_files, categories=preselected_categories)
        candidates_files = filtered if filtered else raw_files
        options = [(f"{Path(f).name}  [{f}]", f) for f in candidates_files]
        paths = candidates_files

    selector = widgets.SelectMultiple(
        options=options,
        rows=min(18, max(5, len(options))),
        description="Modelos:",
        layout=widgets.Layout(width="100%", height="420px"),
    )
    select_all = widgets.Button(description="Selecionar todos")
    clear_all = widgets.Button(description="Limpar")
    confirm = widgets.Button(description="Confirmar seleção", button_style="success")
    output = widgets.Output()

    def all_click(_): selector.value = tuple(paths)
    def clear_click(_): selector.value = tuple()
    def confirm_click(_):
        with output:
            clear_output(wait=True)
            chosen = list(selector.value)
            if not chosen:
                print("Nenhum arquivo selecionado.")
            else:
                print("Selecionados:")
                for item in chosen: print("  -", item)
                print(f"Total: {len(chosen)} arquivo(s)")
            # Armazena em variável global persistente do notebook
            globals()["SELECTED_FILES"] = chosen

    select_all.on_click(all_click)
    clear_all.on_click(clear_click)
    confirm.on_click(confirm_click)
    display(widgets.HTML(f"<b>{dataset}</b> · {len(options)} modelo(s) disponível(is)"))
    display(widgets.HBox([select_all, clear_all, confirm]))
    display(selector, output)
    return selector


In [ ]:
# ESCOLHA MANUAL DOS MODELOS (Apenas os selecionados serão baixados para o SSD)
CATEGORIES = ["checkpoints", "diffusion_models", "loras", "vae", "text_encoders", "clip", "controlnet", "upscale_models", "video_models", "embeddings"]
SELECTED_FILES = []  # será preenchido ao clicar em 'Confirmar seleção'
selector = choose_dataset_files(DATASET, preselected_categories=CATEGORIES)


In [ ]:
# Sincronização seletiva dos modelos escolhidos
# SELECTED_FILES é preenchido pelo callback do botão 'Confirmar seleção'
if not SELECTED_FILES:
    raise ValueError("Nenhum modelo selecionado. Clique em 'Confirmar seleção' na célula anterior.")

stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=MODELS_DIR,
    selected_files=SELECTED_FILES,
    force=False,
)
print(json.dumps(stats, indent=2, ensure_ascii=False))


In [ ]:
# Inicia ComfyUI em background com output local no SSD e valida a API
from comfyui_setup import start_comfyui, health_check

COMFYUI_PORT = 8188
proc = start_comfyui(
    comfyui_dir=COMFYUI_DIR,
    host="0.0.0.0",
    port=COMFYUI_PORT,
    output_dir=OUTPUT_DIR,
)

if not health_check("127.0.0.1", COMFYUI_PORT, timeout=90):
    raise RuntimeError("ComfyUI iniciou, mas não respondeu ao health check.")

print(f"✅ ComfyUI OK em http://127.0.0.1:{COMFYUI_PORT}")
print(f"✅ Outputs locais em: {OUTPUT_DIR}")
print("PID:", proc.pid)


In [ ]:
# Push inicial condicional: verifica se há outputs ou logs locais e oferece/executa sync
from kaggle_drive_sync import sync_outputs

existing_outputs = list(OUTPUT_DIR.glob("*")) if OUTPUT_DIR.exists() else []
existing_outputs = [f for f in existing_outputs if f.is_file()]
log_file = COMFYUI_DIR / "comfyui.log"
has_logs = log_file.exists() and log_file.stat().st_size > 0

if existing_outputs or has_logs:
    print(f"[INFO] Arquivos detectados: {len(existing_outputs)} output(s), log={has_logs}")
    cats = []
    if existing_outputs: cats.append("outputs")
    if has_logs: cats.append("logs")
    try:
        sync_res = sync_outputs(action="push", categories=cats, local_outputs=OUTPUT_DIR, drive_base=DRIVE_BASE, env="kaggle")
        print(f"[INFO] Push inicial concluído: {sync_res['synced']} enviado(s), {sync_res['skipped']} inalterado(s)")
    except Exception as e:
        print(f"[WARN] Push inicial não pôde ser executado: {e}")
else:
    print("[INFO] Nenhum arquivo em output/ ou log para envio no momento. Pronto para gerar!")


## Próximo passo

O ComfyUI está pronto e rodando localmente no SSD do Kaggle. Gere suas imagens normalmente pela interface ou API.

Para sincronizar os outputs gerados, workflows e logs com o Google Drive a qualquer momento, abra e execute o notebook manual:
**`09_sync_outputs.ipynb`**
